In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

**Step 1: Define Model Architecture**

In [ ]:
class TinyLLM(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, num_layers=6, dim_feedforward=2048, max_seq_length=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.max_seq_length = max_seq_length
        
        # Token and position embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_length, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Transformer encoder layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoderLayer(encoder_layer, num_layers=num_layers)
        
        # Output projection
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, x, mask=None):
        """
        Args:
            x: Input token ids [batch_size, seq_length]
            mask: Attention mask [batch_size, seq_length]
        """
        batch_size, seq_length = x.size()
        
        # Create position indices
        positions = torch.arange(0, seq_length, device=x.device).unsqueeze(0).expand(batch_size, -1)
        
        # Embeddings
        token_emb = self.token_embedding(x)
        pos_emb = self.position_embedding(positions)
        x = self.dropout(token_emb + pos_emb)
        
        # Create attention mask (1 for valid, 0 for padding)
        if mask is None:
            mask = torch.ones(batch_size, seq_length, device=x.device, dtype=torch.bool)
        
        # Convert to attention mask format (False for valid, True for padding)
        attn_mask = ~mask
        
        # Transformer encoder
        x = self.transformer(x, src_key_padding_mask=attn_mask)
        
        # Layer norm and output projection
        x = self.ln_f(x)
        logits = self.head(x)
        
        
        """
        Returns:
            logits: [batch_size, seq_length, vocab_size]
        """
        return logits